# kc_house_data – 2D-Karten mit Folium & PyDeck

Ein Notebook, zwei Karten-Bibliotheken für denselben Datensatz (King County Hausverkäufe).

**Jeder** der ~21.600 Datenpunkte wird einzeln an seiner Position dargestellt – keine Cluster, keine Heatmap, flach 2D. Farbe = Preis (blau = günstig → rot = teuer).

- **Folium + Leaflet** – einzelne CircleMarker
- **PyDeck (deck.gl)** – ScatterplotLayer, Draufsicht

Jede Karte wird direkt in der Zelle angezeigt und zusätzlich als `viz/kc_*.html` gespeichert.

## 1 · Setup: Pakete (bei Bedarf einmalig ausführen)

In [ ]:
# Einmalig ausführen, falls noch nicht installiert. Danach Zelle überspringen.
%pip install folium pydeck

## 2 · Daten laden & bereinigen

In [ ]:
from pathlib import Path
import pandas as pd

CSV = Path('kc_house_data.csv')
OUT = Path('viz'); OUT.mkdir(exist_ok=True)

df = pd.read_csv(CSV)
# nur plausible Koordinaten im Raum King County
df = df[df['lat'].between(47.0, 48.0) & df['long'].between(-122.6, -121.2)].reset_index(drop=True)
print(f'{len(df):,} Häuser')
df[['price', 'bedrooms', 'bathrooms', 'sqft_living', 'yr_built', 'lat', 'long']].head()

In [ ]:
def price_to_rgb(prices):
    """Preis -> Farbe (blau=günstig .. rot=teuer), robust über Quantile."""
    lo, hi = prices.quantile(0.02), prices.quantile(0.98)
    t = ((prices - lo) / (hi - lo)).clip(0, 1)
    r = (255 * t).round().astype(int)
    g = (60 + 100 * (1 - (2 * t - 1).abs())).round().astype(int)
    b = (255 * (1 - t)).round().astype(int)
    return r, g, b

# RGB + Hex je Datenpunkt einmal vorberechnen
R, G, B = price_to_rgb(df['price'])
df['r'], df['g'], df['b'] = R, G, B
df['hex'] = ['#%02x%02x%02x' % (ri, gi, bi) for ri, gi, bi in zip(R, G, B)]

## 3 · Folium + Leaflet
Jeder Datenpunkt als einzelner CircleMarker an seiner exakten Position, eingefärbt nach Preis. Klick auf einen Punkt zeigt Details.

> Hinweis: ~21.600 einzelne Marker ergeben eine große HTML-Datei und rendern im Browser etwas langsamer als eine geclusterte Karte – das ist der Preis dafür, wirklich jeden Punkt einzeln zu zeigen.

In [ ]:
import folium
from folium import Figure

# Größe der Karte hier einstellen (Pixel):
fig = Figure(width=1100, height=800)

m = folium.Map(location=[df['lat'].mean(), df['long'].mean()],
               zoom_start=10, tiles='CartoDB positron')
fig.add_child(m)

for row in df.itertuples():
    folium.CircleMarker(
        [row.lat, row.long], radius=1, weight=0,
        fill=True, fill_color=row.hex, fill_opacity=1,
        popup=folium.Popup(
            f'<b>${row.price:,.0f}</b><br>{row.bedrooms} Bett / {row.bathrooms} Bad'
            f'<br>{row.sqft_living} sqft · Bj {row.yr_built}', max_width=220)
    ).add_to(m)

m.save(OUT / 'kc_folium.html')
fig

## 4 · PyDeck (deck.gl)
Alle Punkte als GPU-Scatterplot in Draufsicht (2D, `pitch=0`). Farbe = Preis, Hover = Tooltip.

In [ ]:
import pydeck as pdk

dfd = df.assign(price_k=(df['price'] / 1000).round())

scatter = pdk.Layer('ScatterplotLayer', data=dfd, get_position='[long, lat]',
                    get_fill_color='[r, g, b, 180]', get_radius=50,
                    radius_min_pixels=2, pickable=True)

view = pdk.ViewState(latitude=df['lat'].mean(), longitude=df['long'].mean(), zoom=9.5, pitch=0)
deck = pdk.Deck(layers=[scatter], initial_view_state=view,
                width='100%', height=800,  # Größe der Karte hier einstellen
                map_provider='carto', map_style='light',
                tooltip={'text': '${price_k}k\n{bedrooms} Bett / {bathrooms} Bad\n{sqft_living} sqft'})
deck.to_html(str(OUT / 'kc_pydeck.html'), open_browser=False)
deck

## 5 · Fokus: Waterfront **und** Preis über Median
Hebt die Häuser hervor, die **beide** Bedingungen erfüllen – `waterfront == 1` **und** `price > Median` (rot, groß). Alle übrigen Häuser liegen dezent grau im Hintergrund, damit die Lage im Kontext sichtbar ist.

In [ ]:
import pydeck as pdk

med = df['price'].median()
mask = (df['waterfront'] == 1) & (df['price'] > med)
hl = df[mask].assign(price_k=(df.loc[mask, 'price'] / 1000).round())
print(f'Median-Preis: ${med:,.0f}  ·  {len(hl)} Häuser mit Waterfront & Preis > Median')

# Hintergrund: alle Häuser dezent grau (Kontext)
bg = pdk.Layer('ScatterplotLayer', data=df, get_position='[long, lat]',
               get_fill_color='[160, 160, 160, 55]', get_radius=40, radius_min_pixels=1)

# Highlight: Waterfront UND Preis über Median, kräftig rot und größer
hi = pdk.Layer('ScatterplotLayer', data=hl, get_position='[long, lat]',
               get_fill_color='[220, 30, 30, 230]', get_radius=180,
               radius_min_pixels=5, pickable=True)

view = pdk.ViewState(latitude=df['lat'].mean(), longitude=df['long'].mean(), zoom=9, pitch=0)
deck_hl = pdk.Deck(layers=[bg, hi], initial_view_state=view,
                   width='100%', height=800,  # Größe hier einstellen
                   map_provider='carto', map_style='light',
                   tooltip={'text': '${price_k}k · Waterfront\n{bedrooms} Bett / {bathrooms} Bad · {sqft_living} sqft'})
deck_hl.to_html(str(OUT / 'kc_waterfront_ueber_median.html'), open_browser=False)
deck_hl